# 🛡️ ARGUS Security Graph & Risk Intelligence Lab

This notebook serves as a research and prototyping environment for the **ARGUS Security Graph** (Phase 3.5 of the ARGUS OS roadmap).

It simulates how edge surveillance events (vehicle tracking, license plate detections, loitering events, restricted zone intrusions, and fire/smoke alarms) evolve into **Entities**, **Observations**, **Relationships**, **Dynamic Risk Scores**, and **Graph-Based Intelligence Insights**.

### 📋 Lab Sections
1. **Setup & Installation**: Install dependencies (`networkx`, `plotly`, `pyvis`, `pandas`, `numpy`, `matplotlib`).
2. **Event Simulator**: Stream generator for mock security observations.
3. **Entity Resolution Engine**: Class-based resolver that merges duplicate observations into clean Entity profiles.
4. **Relationship Graph Constructor**: Connect entities, events, spatial locations, and relationships using NetworkX.
5. **Dynamic Risk Engine**: Score risk dynamically for entities and locations, featuring temporal decay and graph-based propagation (guilt by association).
6. **Timeline Analysis**: Chronologically trace the movement history of vehicles and individuals.
7. **Interactive Visualizations**: Render the relationship network using Plotly and browser-interactive PyVis graphs.
8. **Graph Intelligence Queries**: Discover visited locations, entities linked to alarms, and suspicious casing/speeding patterns.
9. **Neo4j Export**: Export nodes and relationships directly to CSV with Cypher loading templates.

In [ ]:
# Install dependencies (pandas, networkx, plotly, pyvis, numpy, matplotlib)
!pip install pandas networkx plotly pyvis numpy matplotlib

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import networkx as nx
import plotly.graph_objects as go
from pyvis.network import Network
from datetime import datetime, timedelta
import re
import os
import IPython
from IPython.display import HTML, display

print("All libraries imported successfully!")

--- 
## 1. Security Event Simulator

Generate realistic security events simulating observations across multiple edge cameras. We will model a realistic sequence where a suspicious vehicle enters Gate A, is seen at the Loading Dock, a tracked person gets out and loiters, triggers a restricted zone intrusion and a fire alarm, and then exits the site.

In [ ]:
import random
from datetime import datetime, timedelta

class SecurityEventSimulator:
    def __init__(self, start_time=None, environment="Warehouse"):
        self.start_time = start_time or datetime.now()
        self.environment = environment
        
        self.locations = {
            "Supermarket": ["Main Entrance", "Aisle 4", "Self Checkout", "Loading Bay", "Parking Lot"],
            "Estate": ["Main Gate", "Visitor Parking", "Block A Lobby", "Pool Area", "Service Gate"],
            "School Campus": ["Library Entrance", "Student Union", "Dormitory B", "North Parking", "Science Building"],
            "Market": ["North Gate", "Central Plaza", "Food Stalls", "Delivery Area", "West Exit"],
            "Community Security": ["Checkpoint Alpha", "Park Zone", "Street C", "Community Center"],
            "Warehouse": ["Gate A", "Loading Dock", "Warehouse Alpha", "Gate B", "Storage Area"]
        }
        
    def generate_scenario_events(self, num_events=20):
        events = []
        base = self.start_time
        locs = self.locations.get(self.environment, self.locations["Warehouse"])
        
        # Scenarios and entities
        plates = ["ABC-123-AA", "XYZ-987-BB", "KIP-5567", "DEF-456-CC", "GHI-789-DD"]
        track_ids = ["TRK_201", "TRK_202", "TRK_203", "TRK_204"]
        
        # We will create a cohesive narrative mixed with random noise
        # Narrative 1: Suspicious vehicle and casing
        events.append({
            "event_id": "EVT_001",
            "timestamp": (base + timedelta(minutes=1)).strftime("%Y-%m-%d %H:%M:%S"),
            "event_type": "VEHICLE_DETECTION",
            "location": locs[0], # Entrance
            "details": {"license_plate": plates[0], "vehicle_type": "sedan", "color": "dark blue"}
        })
        
        events.append({
            "event_id": "EVT_002",
            "timestamp": (base + timedelta(minutes=5)).strftime("%Y-%m-%d %H:%M:%S"),
            "event_type": "VEHICLE_DETECTION",
            "location": locs[1],
            "details": {"license_plate": plates[0].replace("-", ""), "vehicle_type": "sedan", "color": "blue"}
        })
        
        events.append({
            "event_id": "EVT_003",
            "timestamp": (base + timedelta(minutes=7)).strftime("%Y-%m-%d %H:%M:%S"),
            "event_type": "LOITERING",
            "location": locs[1],
            "details": {"person_track_id": track_ids[0], "clothing": "black hoodie, blue jeans", "duration_seconds": 180}
        })
        
        events.append({
            "event_id": "EVT_004",
            "timestamp": (base + timedelta(minutes=10)).strftime("%Y-%m-%d %H:%M:%S"),
            "event_type": "LOITERING",
            "location": locs[2],
            "details": {"person_track_id": track_ids[0], "clothing": "black hoodie", "duration_seconds": 320}
        })
        
        events.append({
            "event_id": "EVT_005",
            "timestamp": (base + timedelta(minutes=15)).strftime("%Y-%m-%d %H:%M:%S"),
            "event_type": "INTRUSION",
            "location": locs[2],
            "details": {"person_track_id": track_ids[0], "zone_id": "Restricted Zone A"}
        })
        
        events.append({
            "event_id": "EVT_006",
            "timestamp": (base + timedelta(minutes=17)).strftime("%Y-%m-%d %H:%M:%S"),
            "event_type": "FIRE_SMOKE",
            "location": locs[2],
            "details": {"sensor_id": "SMK_W1", "severity": "HIGH"}
        })
        
        events.append({
            "event_id": "EVT_007",
            "timestamp": (base + timedelta(minutes=22)).strftime("%Y-%m-%d %H:%M:%S"),
            "event_type": "VEHICLE_DETECTION",
            "location": locs[3],
            "details": {"license_plate": plates[1], "vehicle_type": "van", "color": "white"}
        })
        
        events.append({
            "event_id": "EVT_008",
            "timestamp": (base + timedelta(minutes=25)).strftime("%Y-%m-%d %H:%M:%S"),
            "event_type": "VEHICLE_DETECTION",
            "location": locs[3],
            "details": {"license_plate": plates[0], "vehicle_type": "sedan", "color": "blue"}
        })
        
        # Add random noise events
        current_time = base + timedelta(minutes=2)
        for i in range(num_events - 8):
            evt_type = random.choice(["VEHICLE_DETECTION", "LOITERING"])
            loc = random.choice(locs)
            current_time += timedelta(minutes=random.randint(1, 10))
            
            if evt_type == "VEHICLE_DETECTION":
                plate = random.choice(plates[2:])
                events.append({
                    "event_id": f"EVT_N{i:03d}",
                    "timestamp": current_time.strftime("%Y-%m-%d %H:%M:%S"),
                    "event_type": evt_type,
                    "location": loc,
                    "details": {"license_plate": plate, "vehicle_type": "suv", "color": "silver"}
                })
            else:
                track = random.choice(track_ids[1:])
                events.append({
                    "event_id": f"EVT_N{i:03d}",
                    "timestamp": current_time.strftime("%Y-%m-%d %H:%M:%S"),
                    "event_type": evt_type,
                    "location": loc,
                    "details": {"person_track_id": track, "clothing": "red shirt", "duration_seconds": random.randint(60, 300)}
                })
                
        # Sort chronologically
        events = sorted(events, key=lambda x: datetime.strptime(x["timestamp"], "%Y-%m-%d %H:%M:%S"))
        
        # Re-assign IDs properly
        for i, e in enumerate(events):
            e["event_id"] = f"EVT_{i+1:03d}"
            
        return events

# Run simulation
print("Using Supermarket Environment Simulator...")
sim = SecurityEventSimulator(environment="Supermarket")
events = sim.generate_scenario_events(num_events=20)
df_events = pd.DataFrame([{
    "ID": e["event_id"],
    "Timestamp": e["timestamp"],
    "Type": e["event_type"],
    "Location": e["location"],
    "Details": str(e["details"])
} for e in events])
display(df_events)

--- 
## 1b. External Data Ingestion
Import events generated by the live camera notebooks (Intrusion, Loitering, Fire/Smoke, LPR) via CSV or JSON.\n

In [ ]:
import json

def import_events_from_file(filepath):
    """
    Import events from a JSON or CSV file generated by external camera notebooks.
    """
    if filepath.endswith('.csv'):
        df = pd.read_csv(filepath)
        # Parse Details column back to dictionary if needed
        # Assume standard format matching the dataframe output
        events_list = []
        for i, row in df.iterrows():
            details = row["Details"]
            if isinstance(details, str):
                try:
                    # simplistic eval for stringified dicts
                    import ast
                    details = ast.literal_eval(details)
                except:
                    details = {}
            events_list.append({
                "event_id": row["ID"],
                "timestamp": str(row["Timestamp"]),
                "event_type": row["Type"],
                "location": row["Location"],
                "details": details
            })
        return events_list
    elif filepath.endswith('.json'):
        with open(filepath, 'r') as f:
            return json.load(f)
    else:
        raise ValueError("Unsupported file format. Please provide .csv or .json")

# Uncomment and run to import your external data
# events = import_events_from_file("my_external_events.json")
# print(f"Imported {len(events)} events from external source.")\n

--- 
## 2. Entity Resolution Engine

Automatically merge repeated observations into unified Entity profiles. This step matches license plates (incorporating Levenshtein character offsets for OCR readability failures) and tracks persons based on recurring camera identifiers or clothing description attributes.

In [ ]:
class EntityResolver:
    def __init__(self):
        self.entities = {} # entity_id -> entity_profile
        
    def _normalize_plate(self, plate):
        if not plate:
            return ""
        return re.sub(r'[^A-Za-z0-9]', '', plate).upper()
        
    def _plates_match(self, plate1, plate2):
        p1 = self._normalize_plate(plate1)
        p2 = self._normalize_plate(plate2)
        if not p1 or not p2:
            return False
        if p1 == p2:
            return True
        # Allow a mismatch of 1 character representing camera OCR scanning errors
        if len(p1) == len(p2):
            diffs = sum(1 for c1, c2 in zip(p1, p2) if c1 != c2)
            if diffs <= 1:
                return True
        return False

    def resolve_entities(self, events):
        resolved_mappings = {} # event_id -> entity_id
        
        for evt in events:
            evt_id = evt["event_id"]
            details = evt["details"]
            
            # VEHICLE RESOLUTION
            if "license_plate" in details:
                plate = details["license_plate"]
                found_id = None
                
                for ent_id, ent in self.entities.items():
                    if ent["type"] == "VEHICLE" and self._plates_match(ent["details"].get("license_plate"), plate):
                        found_id = ent_id
                        break
                        
                if found_id:
                    self.entities[found_id]["observations"].append(evt)
                    # Merge details, saving clean version (no dashes)
                    normalized_curr = self._normalize_plate(self.entities[found_id]["details"]["license_plate"])
                    normalized_new = self._normalize_plate(plate)
                    if len(normalized_new) >= len(normalized_curr):
                        self.entities[found_id]["details"]["license_plate"] = plate.replace("-", "").upper()
                    resolved_mappings[evt_id] = found_id
                else:
                    new_id = f"VEH_{len(self.entities)+1:03d}"
                    self.entities[new_id] = {
                        "entity_id": new_id,
                        "type": "VEHICLE",
                        "label": f"Vehicle ({plate.replace('-', '').upper()})",
                        "details": {"license_plate": plate, "vehicle_type": details.get("vehicle_type"), "color": details.get("color")},
                        "observations": [evt]
                    }
                    resolved_mappings[evt_id] = new_id
                    
            # PERSON RESOLUTION
            elif "person_track_id" in details:
                track_id = details["person_track_id"]
                found_id = None
                
                for ent_id, ent in self.entities.items():
                    if ent["type"] == "PERSON" and ent["details"].get("person_track_id") == track_id:
                        found_id = ent_id
                        break
                        
                # Merge if clothing features overlap
                if not found_id and "clothing" in details:
                    clothing = details["clothing"]
                    for ent_id, ent in self.entities.items():
                        if ent["type"] == "PERSON":
                            ent_clothing = ent["details"].get("clothing", "")
                            c1_words = set(re.findall(r'\w+', clothing.lower()))
                            c2_words = set(re.findall(r'\w+', ent_clothing.lower()))
                            common = c1_words.intersection(c2_words)
                            if len(common) >= 2 or (len(common) >= 1 and "hoodie" in common):
                                found_id = ent_id
                                break
                                
                if found_id:
                    self.entities[found_id]["observations"].append(evt)
                    resolved_mappings[evt_id] = found_id
                else:
                    new_id = f"PER_{len(self.entities)+1:03d}"
                    self.entities[new_id] = {
                        "entity_id": new_id,
                        "type": "PERSON",
                        "label": f"Person (Track {track_id})",
                        "details": {"person_track_id": track_id, "clothing": details.get("clothing")},
                        "observations": [evt]
                    }
                    resolved_mappings[evt_id] = new_id
                    
        return self.entities, resolved_mappings

# Run Resolution
resolver = EntityResolver()
resolved_entities, resolved_mappings = resolver.resolve_entities(events)

print("--- RESOLVED ENTITIES REGISTERED ---")
for ent_id, ent in resolved_entities.items():
    print(f"ID: {ent_id} | Type: {ent['type']} | Name: {ent['label']} | Observation count: {len(ent['observations'])}")

--- 
## 3. Dynamic Risk Engine

Calculate risk ratings based on event threat profiles, age decay, and structural graph connections. 

### Formula & Decay Details
The risk engine assigns scores on a 0-100 normalized range based on:
1. **Threat Values**: `INTRUSION` (90), `FIRE_SMOKE` (80), `LOITERING` (50), `VEHICLE_DETECTION` (10).
2. **Time Decay**: Decay event scores exponentially: $Risk_{decayed} = Risk_{base} \times e^{-\lambda t}$ (decay rate $\lambda = 0.05$ per minute in simulation).
3. **Graph Association**: Propagate 10% of location threat levels back to visitor entities, and 20% of risks cross-propagated to associated entities (e.g. driver & getaway vehicle sharing spatial overlap).

In [ ]:
class RiskEngine:
    def __init__(self, decay_rate=0.05):
        self.decay_rate = decay_rate
        self.base_weights = {
            "INTRUSION": 90,
            "FIRE_SMOKE": 80,
            "LOITERING": 50,
            "VEHICLE_DETECTION": 10
        }
        
    def calculate_scores(self, entities, events, resolved_mappings, reference_time=None):
        if not reference_time:
            timestamps = [datetime.strptime(evt["timestamp"], "%Y-%m-%d %H:%M:%S") for evt in events]
            reference_time = max(timestamps) if timestamps else datetime.now()
            
        entity_scores = {ent_id: 0.0 for ent_id in entities}
        location_scores = {}
        
        # 1. Base scores & Temporal decay
        for evt in events:
            evt_id = evt["event_id"]
            evt_type = evt["event_type"]
            evt_time = datetime.strptime(evt["timestamp"], "%Y-%m-%d %H:%M:%S")
            loc = evt["location"]
            
            # calculate minutes difference
            min_diff = (reference_time - evt_time).total_seconds() / 60.0
            decay = np.exp(-self.decay_rate * min_diff)
            
            base_val = self.base_weights.get(evt_type, 10)
            decayed_score = base_val * decay
            
            # accumulate on location
            location_scores[loc] = location_scores.get(loc, 0.0) + decayed_score
            
            # accumulate on entity
            ent_id = resolved_mappings.get(evt_id)
            if ent_id:
                entity_scores[ent_id] += decayed_score
                
        # 2. Risk Propagation (guilt by association)
        for ent_id, ent in entities.items():
            # Gain risk from locations visited
            visited_locs = set(obs["location"] for obs in ent["observations"])
            for loc in visited_locs:
                entity_scores[ent_id] += 0.1 * location_scores.get(loc, 0.0)
                
        # Handle spatial-temporal associations (driver & escape vehicle overlap)
        for ent_id1, ent1 in entities.items():
            for ent_id2, ent2 in entities.items():
                if ent_id1 == ent_id2:
                    continue
                for obs1 in ent1["observations"]:
                    t1 = datetime.strptime(obs1["timestamp"], "%Y-%m-%d %H:%M:%S")
                    for obs2 in ent2["observations"]:
                        t2 = datetime.strptime(obs2["timestamp"], "%Y-%m-%d %H:%M:%S")
                        if obs1["location"] == obs2["location"] and abs((t1 - t2).total_seconds()) <= 300:
                            # cross propagate risk
                            entity_scores[ent_id1] += 0.2 * entity_scores[ent_id2]
                            
        # Normalize scores to 0-100 range
        def normalize(scores):
            if not scores:
                return {}
            max_s = max(scores.values())
            if max_s == 0:
                return {k: 0.0 for k in scores}
            return {k: round(min((v / max_s) * 100.0, 100.0), 1) for k, v in scores.items()}
            
        return normalize(entity_scores), normalize(location_scores)

# Calculate risks
risk_engine = RiskEngine()
entity_risk, location_risk = risk_engine.calculate_scores(resolved_entities, events, resolved_mappings)

print("=== DYNAMIC ENTITY RISK RATING ===")
for ent_id, score in sorted(entity_risk.items(), key=lambda x: x[1], reverse=True):
    print(f"- {resolved_entities[ent_id]['label']}: {score} / 100")
    
print("\n=== LOCATION RISK LEVEL ===")
for loc, score in sorted(location_risk.items(), key=lambda x: x[1], reverse=True):
    print(f"- {loc}: {score} / 100")

--- 
## 4. Relationship Graph Construction

Build the relationship graph using `networkx.DiGraph`. Connect Entities, Locations, and raw security Events to reconstruct site operations.

In [ ]:
class SecurityGraph:
    def __init__(self):
        self.G = nx.DiGraph()
        
    def build(self, entities, events, resolved_mappings, entity_risk, location_risk):
        # 1. Add Locations
        for evt in events:
            loc = evt["location"]
            self.G.add_node(loc, type="LOCATION", label=loc, risk_score=location_risk.get(loc, 0.0))
            
        # 2. Add Entities
        for ent_id, ent in entities.items():
            self.G.add_node(ent_id, type=ent["type"], label=ent["label"], risk_score=entity_risk.get(ent_id, 0.0), details=str(ent["details"]))
            
        # 3. Add Events and link to Location
        for evt in events:
            evt_id = evt["event_id"]
            self.G.add_node(evt_id, type="EVENT", label=evt["event_type"], timestamp=evt["timestamp"], location=evt["location"])
            self.G.add_edge(evt_id, evt["location"], relation="OCCURRED_AT")
            
        # 4. Link Entities to triggered events and locations
        for ent_id, ent in entities.items():
            for obs in ent["observations"]:
                evt_id = obs["event_id"]
                self.G.add_edge(ent_id, evt_id, relation="TRIGGERED")
                self.G.add_edge(ent_id, obs["location"], relation="SEEN_AT", timestamp=obs["timestamp"])
                
        # 5. Link Associated Entities (Spatial-Temporal overlap)
        for ent_id1, ent1 in entities.items():
            for ent_id2, ent2 in entities.items():
                if ent_id1 >= ent_id2:
                    continue
                for obs1 in ent1["observations"]:
                    t1 = datetime.strptime(obs1["timestamp"], "%Y-%m-%d %H:%M:%S")
                    for obs2 in ent2["observations"]:
                        t2 = datetime.strptime(obs2["timestamp"], "%Y-%m-%d %H:%M:%S")
                        if obs1["location"] == obs2["location"] and abs((t1 - t2).total_seconds()) <= 300:
                            self.G.add_edge(ent_id1, ent_id2, relation="ASSOCIATED_WITH", reason="Spatial-Temporal Overlap")
                            self.G.add_edge(ent_id2, ent_id1, relation="ASSOCIATED_WITH", reason="Spatial-Temporal Overlap")
                            
        return self.G

# Build Graph
graph_builder = SecurityGraph()
G = graph_builder.build(resolved_entities, events, resolved_mappings, entity_risk, location_risk)
print(f"Security Graph loaded with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

--- 
## 5. Timeline Analysis & Movement History

Trace entity traversals chronologically to analyze exact movement flows over time across edge nodes.

In [ ]:
def get_entity_timeline(graph, entity_id):
    if entity_id not in graph:
        return []
    
    timeline = []
    for u, v, data in graph.out_edges(entity_id, data=True):
        if data.get("relation") == "SEEN_AT":
            timeline.append({
                "timestamp": data.get("timestamp"),
                "location": v
            })
            
    return sorted(timeline, key=lambda x: datetime.strptime(x["timestamp"], "%Y-%m-%d %H:%M:%S"))

# Render history timelines for all entities in the graph
entity_nodes = [n for n, d in G.nodes(data=True) if d.get("type") in ["VEHICLE", "PERSON"]]
for ent_id in entity_nodes:
    print(f"\n=== MOVEMENT TIMELINE FOR {G.nodes[ent_id]['label'].upper()} ===")
    for record in get_entity_timeline(G, ent_id):
        print(f"- {record['timestamp']}: Spotted at {record['location']}")

--- 
## 6. Interactive Visualizations

### 6.1 Plotly Graph Visualization (Color Scale by Risk)
Plotly provides a scatter plot network where node color correlates directly with the node's risk rating.

In [ ]:
def draw_plotly_graph(graph):
    pos = nx.spring_layout(graph, k=0.55, seed=42)
    
    # Edges Coordinates
    edge_x = []
    edge_y = []
    for edge in graph.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])
        
    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        line=dict(width=1.5, color='#4b5563'),
        hoverinfo='none',
        mode='lines'
    )
    
    # Nodes Coordinates
    node_x = []
    node_y = []
    node_text = []
    node_color = []
    node_size = []
    node_labels = []
    
    for node in graph.nodes():
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
        
        data = graph.nodes[node]
        ntype = data.get("type", "LOCATION")
        risk = data.get("risk_score", 0.0)
        label = data.get("label", node)
        
        # Clean short labels to print over node
        short_label = label.split(" ")[0] if len(label) > 10 else label
        node_labels.append(short_label)
        
        # Hover Text
        hover_html = f"<b>{label}</b><br>Type: {ntype}<br>Risk Rating: {risk}"
        if "timestamp" in data:
            hover_html += f"<br>Event Time: {data['timestamp']}"
        node_text.append(hover_html)
        
        # Color scale based on Risk
        if ntype == "EVENT":
            # Hardcode event alert weight representations
            base_alert_risk = 50.0
            if label == "INTRUSION": base_alert_risk = 90.0
            elif label == "FIRE_SMOKE": base_alert_risk = 80.0
            elif label == "VEHICLE_DETECTION": base_alert_risk = 10.0
            node_color.append(base_alert_risk)
            node_size.append(14)
        else:
            node_color.append(risk)
            node_size.append(18 + risk * 0.15)
            
    node_trace = go.Scatter(
        x=node_x, y=node_y,
        mode='markers+text',
        text=node_labels,
        textposition="top center",
        hoverinfo='text',
        hovertext=node_text,
        marker=dict(
            showscale=True,
            colorscale='YlOrRd',
            color=node_color,
            size=node_size,
            colorbar=dict(
                thickness=15,
                title='Risk Score',
                xanchor='left',
                titleside='right'
            ),
            line=dict(width=2, color='#111827')
        )
    )
    
    fig = go.Figure(data=[edge_trace, node_trace],
                 layout=go.Layout(
                    title='<b>ARGUS Security Threat Graph Network Map</b>',
                    titlefont_size=16,
                    showlegend=False,
                    hovermode='closest',
                    margin=dict(b=20,l=5,r=5,t=40),
                    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                    plot_bgcolor='#111827',
                    paper_bgcolor='#111827',
                    font=dict(color='#f3f4f6')
                 ))
    fig.show()

draw_plotly_graph(G)

### 6.2 PyVis interactive browser animation
Generate a fully interactive HTML network canvas that renders dynamically directly inside your Colab output. You can click, zoom, drag nodes, and hover to see attributes.

In [ ]:
def create_pyvis_graph(graph, filename="security_graph.html"):
    # Create canvas
    net = Network(height="550px", width="100%", bgcolor="#111827", font_color="#f3f4f6", notebook=True, directed=True, cdn_resources='in_line')
    
    color_map = {
        "LOCATION": "#10b981", # Emerald Green
        "EVENT": "#ef4444",    # Crimson Red
        "VEHICLE": "#3b82f6",  # Royal Blue
        "PERSON": "#f59e0b"    # Amber Orange
    }
    
    # Add Nodes
    for node, data in graph.nodes(data=True):
        ntype = data.get("type", "LOCATION")
        risk = data.get("risk_score", 0.0)
        label = data.get("label", node)
        
        color = color_map.get(ntype, "#6b7280")
        size = 16 + (risk * 0.2)
        title_text = f"<b>{label}</b><br>Type: {ntype}<br>Threat Score: {risk}"
        
        net.add_node(node, label=label, color=color, size=size, title=title_text)
        
    # Add Edges
    for u, v, data in graph.edges(data=True):
        relation = data.get("relation", "")
        color = "#9ca3af"
        if relation == "TRIGGERED": color = "#f87171"
        elif relation == "ASSOCIATED_WITH": color = "#fbbf24"
        
        net.add_edge(u, v, label=relation, color=color)
        
    # Render graph to HTML file
    net.save_graph(filename)
    print(f"Graph successfully saved to: {filename}")

# Save interactive HTML
create_pyvis_graph(G)

# Render directly in Google Colab output
display(HTML("security_graph.html"))

--- 
## 7. Graph Intelligence Queries

Run forensic queries to find traversal paths, link entities directly to high-threat alerts, and identify suspicious threat behavior patterns.

In [ ]:
# Query 1: Find all locations visited by a vehicle
def find_locations_visited_by_vehicle(graph, vehicle_id):
    locations = []
    if vehicle_id in graph:
        for u, v, data in graph.out_edges(vehicle_id, data=True):
            if data.get("relation") == "SEEN_AT":
                locations.append((v, data.get("timestamp")))
    return sorted(locations, key=lambda x: datetime.strptime(x[1], "%Y-%m-%d %H:%M:%S"))

vehicles = [n for n, d in G.nodes(data=True) if d.get("type") == "VEHICLE"]
if vehicles:
    target_veh = vehicles[0]
    print(f"Locations visited by {G.nodes[target_veh]['label']}:")
    for loc, t in find_locations_visited_by_vehicle(G, target_veh):
        print(f" - {loc} at {t}")

In [ ]:
# Query 2: Find all entities linked to intrusion events
def find_entities_linked_to_intrusions(graph):
    linked_entities = []
    intrusion_nodes = [n for n, d in graph.nodes(data=True) if d.get("type") == "EVENT" and d.get("label") == "INTRUSION"]
    
    for in_node in intrusion_nodes:
        predecessors = list(graph.predecessors(in_node))
        for pred in predecessors:
            pred_data = graph.nodes[pred]
            linked_entities.append({
                "Entity ID": pred,
                "Label": pred_data.get("label"),
                "Type": pred_data.get("type"),
                "Event ID": in_node,
                "Location": graph.nodes[in_node].get("location"),
                "Timestamp": graph.nodes[in_node].get("timestamp")
            })
    return linked_entities

df_intrusions = pd.DataFrame(find_entities_linked_to_intrusions(G))
display(df_intrusions)

In [ ]:
# Query 3: Find suspicious activity patterns (casing / loitering before intrusion, speed violations)
def detect_suspicious_patterns(graph):
    patterns = []
    
    # Pattern A: Loitering followed by Intrusion at the same location within 30 minutes (Casing)
    loitering_nodes = [n for n, d in graph.nodes(data=True) if d.get("type") == "EVENT" and d.get("label") == "LOITERING"]
    for ln in loitering_nodes:
        loc = graph.nodes[ln].get("location")
        t_loiter = datetime.strptime(graph.nodes[ln].get("timestamp"), "%Y-%m-%d %H:%M:%S")
        
        persons = [p for p in graph.predecessors(ln) if graph.nodes[p].get("type") == "PERSON"]
        for person in persons:
            for edge in graph.out_edges(person):
                target = edge[1]
                t_node = graph.nodes[target]
                if t_node.get("type") == "EVENT" and t_node.get("label") == "INTRUSION" and t_node.get("location") == loc:
                    t_intrusion = datetime.strptime(t_node.get("timestamp"), "%Y-%m-%d %H:%M:%S")
                    time_diff = (t_intrusion - t_loiter).total_seconds()
                    if 0 < time_diff <= 1800:
                        patterns.append({
                            "Pattern Class": "LOITERING_BEFORE_INTRUSION (CASING)",
                            "Entity": graph.nodes[person].get("label"),
                            "Location": loc,
                            "Time Point 1": t_loiter.strftime("%Y-%m-%d %H:%M:%S"),
                            "Time Point 2": t_intrusion.strftime("%Y-%m-%d %H:%M:%S"),
                            "Summary": f"Entity loitered before breaking in. Deliberate offset of {time_diff/60:.1f} mins."
                        })
                        
    # Pattern B: Rapid Traversal (Suspiciously fast travel between distant location gates)
    vehicles = [n for n, d in graph.nodes(data=True) if d.get("type") == "VEHICLE"]
    for veh in vehicles:
        seen_edges = [edge for edge in graph.out_edges(veh, data=True) if edge[2].get("relation") == "SEEN_AT"]
        seen_edges = sorted(seen_edges, key=lambda x: datetime.strptime(x[2].get("timestamp"), "%Y-%m-%d %H:%M:%S"))
        
        for i in range(len(seen_edges) - 1):
            loc1 = seen_edges[i][1]
            t1 = datetime.strptime(seen_edges[i][2].get("timestamp"), "%Y-%m-%d %H:%M:%S")
            loc2 = seen_edges[i+1][1]
            t2 = datetime.strptime(seen_edges[i+1][2].get("timestamp"), "%Y-%m-%d %H:%M:%S")
            
            if loc1 != loc2:
                time_diff_sec = (t2 - t1).total_seconds()
                # Flag travel between gate segments happening in under 5 minutes
                if time_diff_sec < 300:
                    patterns.append({
                        "Pattern Class": "RAPID_TRAVERSAL (SPEEDING/TAILGATING)",
                        "Entity": graph.nodes[veh].get("label"),
                        "Location": f"{loc1} -> {loc2}",
                        "Time Point 1": t1.strftime("%Y-%m-%d %H:%M:%S"),
                        "Time Point 2": t2.strftime("%Y-%m-%d %H:%M:%S"),
                        "Summary": f"Vehicle traversed between distinct cameras in {time_diff_sec/60:.1f} minutes."
                    })
                    
    return patterns

df_patterns = pd.DataFrame(detect_suspicious_patterns(G))
display(df_patterns)

--- 
## 8. Neo4j-Compatible Export

Export Graph structures directly into CSV files for easy loading into a production **Neo4j** server instance.

In [ ]:
def export_to_neo4j_csv(graph, nodes_path="neo4j_nodes.csv", edges_path="neo4j_edges.csv"):
    # 1. Export Nodes
    node_records = []
    for node, data in graph.nodes(data=True):
        node_records.append({
            "id:ID": node,
            "label:LABEL": data.get("type", "LOCATION"),
            "name": data.get("label", node),
            "risk_score": data.get("risk_score", 0.0),
            "timestamp": data.get("timestamp", ""),
            "details": data.get("details", "")
        })
        
    df_nodes = pd.DataFrame(node_records)
    df_nodes.to_csv(nodes_path, index=False)
    
    # 2. Export Edges
    edge_records = []
    for u, v, data in graph.edges(data=True):
        edge_records.append({
            ":START_ID": u,
            ":END_ID": v,
            ":TYPE": data.get("relation", "ASSOCIATED_WITH"),
            "timestamp": data.get("timestamp", ""),
            "reason": data.get("reason", "")
        })
        
    df_edges = pd.DataFrame(edge_records)
    df_edges.to_csv(edges_path, index=False)
    
    print(f"Nodes exported to {nodes_path} ({len(df_nodes)} records)")
    print(f"Edges/Relationships exported to {edges_path} ({len(df_edges)} records)")

# Run CSV Export
export_to_neo4j_csv(G)

### 💡 Cypher Loading Template Script
Import these CSV files into Neo4j using the Cypher query script below:

```cypher
// Load nodes into Neo4j database
LOAD CSV WITH HEADERS FROM 'file:///neo4j_nodes.csv' AS row
MERGE (n:SecurityNode {id: row.`id:ID`})
ON CREATE SET n.label = row.`label:LABEL`,
              n.name = row.name,
              n.risk_score = toFloat(row.risk_score),
              n.timestamp = row.timestamp,
              n.details = row.details;

// Load relations/edges into database
LOAD CSV WITH HEADERS FROM 'file:///neo4j_edges.csv' AS row
MATCH (a:SecurityNode {id: row.`:START_ID`})
MATCH (b:SecurityNode {id: row.`:END_ID`})
CALL apoc.create.relationship(a, row.`:TYPE`, {timestamp: row.timestamp, reason: row.reason}, b)
YIELD rel
RETURN count(rel);
```